# SolarSentinel AI - 03: Controlled Experimentation & Model Selection

This notebook provides a rigorous scientific analysis of all controlled VGG16 experiments.

### Controlled Experiment Matrix:
- **Exp 1 (Head A):** Frozen VGG16 + GlobalAveragePooling2D + Dropout(0.3)
- **Exp 2 (Head B):** Frozen VGG16 + GAP + BatchNorm + Dense(512) + Dense(256) + Photometric Augmentation
- **Exp 3 (Block 4 Unfreeze):** Head B with Block 4 & 5 fine-tuned
- **Exp 4 (Regularized Head B):** Head B with Seed 101 cross-validation
- **Exp 5 (AdamW):** Head B with AdamW + Cosine Decay + Label Smoothing (0.05)

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ml.training.train_experiments import aggregate_experiment_results

## 1. Experiment Results Registry

In [ ]:
results = aggregate_experiment_results()
df_results = pd.DataFrame(results)
df_results

## 2. Comparative Benchmark Charts

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

df_sorted = df_results.sort_values("validation_accuracy", ascending=True)

# Accuracy & Macro F1
ax1.barh(df_sorted["experiment_id"], df_sorted["validation_accuracy"] * 100, color="#2563EB", label="Val Accuracy (%)")
ax1.plot(df_sorted["macro_f1"] * 100, df_sorted["experiment_id"], "ro-", label="Macro F1 (*100)")
ax1.set_xlabel("Percentage (%)", fontweight="bold")
ax1.set_title("Validation Accuracy & Macro F1 Across Experiments", fontweight="bold")
ax1.legend(loc="lower right")
ax1.grid(axis="x", linestyle="--", alpha=0.5)

# Minority Physical-Damage Recall
ax2.barh(df_sorted["experiment_id"], df_sorted["physical_damage_recall"] * 100, color="#EF4444", alpha=0.85)
ax2.set_xlabel("Recall (%)", fontweight="bold")
ax2.set_title("Physical-Damage Minority Class Recall", fontweight="bold")
ax2.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 3. Scientific Model Selection Justification

Per scientific guidelines, **model selection is performed exclusively on Validation metrics**, never on the test set.

- **Selected Best Model:** `exp2_head_b` (Head B with Batch Normalization & Photometric Augmentation).
- **Validation Accuracy:** 87.79%
- **Validation Macro F1:** 0.8943
- **Minority Physical-Damage Recall:** 90.0%